# Analysis Tree Construction

This notebook demonstrates different ways to build analysis trees: nested splits, split_by vs summarize_by vs analyze_by, and expression types.

In [ ]:
import pandas as pd
import numpy as np
from pyMyriad import AnalysisTree

In [ ]:
# Create sample data
np.random.seed(42)
df = pd.DataFrame({
    "Gender": np.random.choice(["M", "F"], 200),
    "Country": np.random.choice(["US", "UK", "Canada"], 200),
    "Age": np.random.randint(18, 70, 200),
    "Income": np.random.normal(50000, 15000, 200),
})

## Nested Splits

Chain multiple splits to create hierarchical subgroups.

In [ ]:
# Split by Gender, then by Country
atree = AnalysisTree()\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .analyze_by(
        mean_income=lambda df: np.mean(df.Income),
        count=lambda df: len(df)
    )

print(atree)
print("\n" + "="*60 + "\n")

result = atree.run(df)
print(result)

## Conditional Splits

Create custom split levels with conditional expressions.

In [ ]:
# Split by age groups using conditional expressions
atree = AnalysisTree()\
    .split_by(
        label="Age Group",
        young=lambda df: df.Age < 40,
        middle=lambda df: (df.Age >= 40) & (df.Age < 60),
        senior=lambda df: df.Age >= 60
    )\
    .analyze_by(
        mean_income=lambda df: np.mean(df.Income),
        count=lambda df: len(df)
    )

print(atree)
print("\n" + "="*60 + "\n")

result = atree.run(df)
print(result)

## analyze_by vs summarize_by

- **analyze_by**: Terminal node - cannot add more splits after it
- **summarize_by**: Non-terminal node - can continue splitting after it

In [ ]:
# Use summarize_by to compute statistics and continue splitting
atree = AnalysisTree()\
    .split_by("df.Gender")\
    .summarize_by(mean_income=lambda df: np.mean(df.Income))\
    .split_by("df.Country")\
    .analyze_by(
        mean_income=lambda df: np.mean(df.Income),
        count=lambda df: len(df)
    )

print(atree)
print("\n" + "="*60 + "\n")

result = atree.run(df)
print(result)

## String Expressions vs Lambda Functions

Both approaches are equivalent - choose based on your preference.

In [ ]:
# Using string expressions
atree_str = AnalysisTree()\
    .split_by("df.Gender")\
    .analyze_by(
        mean_income="np.mean(df.Income)",
        max_age="np.max(df.Age)"
    )

# Using lambda functions
atree_lambda = AnalysisTree()\
    .split_by("df.Gender")\
    .analyze_by(
        mean_income=lambda df: np.mean(df.Income),
        max_age=lambda df: np.max(df.Age)
    )

print("String expression results:")
print(atree_str.run(df))
print("\nLambda function results:")
print(atree_lambda.run(df))

## Complex Analysis with Multiple Levels

Combine everything for sophisticated multi-level analyses.

In [ ]:
atree = AnalysisTree()\
    .split_by("df.Gender")\
    .summarize_by(
        overall_mean="np.mean(df.Income)",
        overall_count="len(df)"
    )\
    .split_by(
        label="Age Group",
        young=lambda df: df.Age < 40,
        older=lambda df: df.Age >= 40
    )\
    .split_by("df.Country")\
    .analyze_by(
        mean_income=lambda df: np.mean(df.Income),
        std_income=lambda df: np.std(df.Income),
        count=lambda df: len(df)
    )

print(atree)
print("\n" + "="*60 + "\n")

result = atree.run(df)
print(result)